# Overflow prediction: tables for per-dataset and merged probe results

This notebook builds two ROC-AUC comparison tables for overflow prediction:

- **Table 1 — Main results**
- **Table 2 — Internal ablation**

Updated behavior:
- each column is configured manually
- supports single datasets and arbitrary merged runs
- supports different experiment names per column
- supports optional feature-only baselines per column

Set the path mappings in the config cell, then run all.


In [17]:
# =========================
# CONFIG
# =========================

COLUMNS = [
    {
        "key": "trivia",
        "label": "TriviaQA",
        "results_dir": "/app/overflow-detection/scripts/data_preprocessing/runs/split_trivia_7b/probe/results",
        "features_path": "/app/overflow-detection/scripts/data_preprocessing/runs/trivia_7b/features_context_metrics.jsonl",
        "experiment_name": "split_trivia_single",
    },
    {
        "key": "squad",
        "label": "SQuAD",
        "results_dir": "/app/overflow-detection/scripts/data_preprocessing/runs/split_squad_7b/probe/results",
        "features_path": "/app/overflow-detection/scripts/data_preprocessing/runs/squad_7b/features_context_metrics.jsonl",
        "experiment_name": "split_squad_single",
    },
    {
        "key": "hotpot",
        "label": "HotpotQA",
        "results_dir": "/app/overflow-detection/scripts/data_preprocessing/runs/split_hotpotqa_7b/probe/results",
        "features_path": "/app/overflow-detection/scripts/data_preprocessing/runs/hotpotqa_7b/features_context_metrics.jsonl",
        "experiment_name": "split_hotpotqa_single",
    },
    # {
    #     "key": "squad_trivia",
    #     "label": "SQuAD + Trivia",
    #     "results_dir": "/app/overflow-detection/scripts/data_preprocessing/runs/split_combined2_7b/probe/results",
    #     "features_path": None,
    #     "experiment_name": "squad_trivia_combined",
    # },
    {
        "key": "all_three",
        "label": "SQuAD + Trivia + Hotpot",
        "results_dir": "/app/overflow-detection/scripts/data_preprocessing/runs/split_combined_7b/probe/results",
        "features_path": "/app/overflow-detection/scripts/data_preprocessing/runs/merged_all_no_llm_7b/features_context_metrics.jsonl",
        "experiment_name": "split_combined_single",
    },
]


In [18]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate


def get_column_cfg(key):
    for col in COLUMNS:
        if col["key"] == key:
            return col
    raise KeyError(key)


COLUMN_ORDER = [c["key"] for c in COLUMNS]
COLUMN_LABELS = {c["key"]: c["label"] for c in COLUMNS}
DATA_COLS = [c["label"] for c in COLUMNS]


def load_dataset(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    df_features = pd.json_normalize(rows, sep=".")
    print(f"Loaded {len(df_features)} rows from {path}")
    return df_features


def load_probing_results(column_key, result_type):
    cfg = get_column_cfg(column_key)
    base_path = cfg["results_dir"]
    experiment_name = cfg["experiment_name"]

    file_mapping = {
        "xrag_no_query": f"probing_results_{experiment_name}_no_query_combined.json",
        "setting1": f"probing_results_setting1_{experiment_name}.json",
        "setting2": f"probing_results_setting2_{experiment_name}.json",
    }

    filepath = os.path.join(base_path, file_mapping[result_type])
    if not os.path.exists(filepath):
        print(f"[warn] Missing {result_type} file for {column_key}: {filepath}")
        return None

    with open(filepath, "r") as f:
        return json.load(f)


def style_auc_table(df, df_means, data_cols):
    style_df = pd.DataFrame('', index=df.index, columns=df.columns)
    for col in data_cols:
        if col not in df_means.columns:
            continue
        vals = df_means[col].dropna()
        if len(vals) == 0:
            continue
        max_val = vals.max()
        second_val = vals.nlargest(2).iloc[1] if len(vals) >= 2 else None
        for i in df.index:
            v = df_means.loc[i, col]
            if pd.isna(v):
                continue
            if v == max_val:
                style_df.loc[i, col] = 'font-weight: bold'
            elif second_val is not None and v == second_val:
                style_df.loc[i, col] = 'text-decoration: underline'
    return style_df


def evaluate_feature_set(X, y, cv_splits=5):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, random_state=42, C=1))
    ])
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
    cv_results = cross_validate(
        pipe, X, y, cv=cv, scoring={'roc_auc': 'roc_auc'}, return_train_score=False
    )
    return {
        'auc_mean': cv_results['test_roc_auc'].mean(),
        'auc_std': cv_results['test_roc_auc'].std(),
    }


def compute_feature_results(df_features):
    saturation_types = ['spec_entropy', 'excess_kurtosis', 'hoyer']

    context_features = ['context_perplexity', 'context_length_chars', 'context_gzip_bits_per_char', 'context_length_tokens']
    preproj_saturation_features = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('preproj_metrics' in col) and (col[-2:] != '_n')]
    postproj_saturation_features = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('postproj_metrics' in col) and (col[-2:] != '_n')]
    prellm_saturation_features = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('postproj_metrics' in col or 'preproj_metrics' in col) and (col[-2:] != '_n')]
    mid_saturation_xrag_only = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('mid_metrics' in col) and ('nonxrag' not in col) and (col[-2:] != '_n')]
    mid_saturation_with_nonxrag = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('nonxrag' in col) and ('mid_group_metrics' in col) and (col[-2:] != '_n') and ('_first' not in col) and ('_last' not in col)] + mid_saturation_xrag_only
    mid_attn_features = [col for col in df_features.columns.values if ('attn_mid' in col)]
    last_saturation_xrag_only = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('last_metrics' in col) and ('nonxrag' not in col) and (col[-2:] != '_n')]
    last_saturation_with_nonxrag = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('last_group_metrics' in col) and ('nonxrag' in col) and (col[-2:] != '_n') and ('_first' not in col) and ('_last' not in col)] + last_saturation_xrag_only
    last_attn_features = [col for col in df_features.columns.values if ('attn_last' in col)]
    postllm_saturation_xrag_only = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('mid_metrics' in col or 'last_metrics' in col) and ('nonxrag' not in col) and (col[-2:] != '_n')]
    postllm_saturation_with_nonxrag = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('mid_group_metrics' in col or 'last_group_metrics' in col) and ('nonxrag' in col) and (col[-2:] != '_n') and ('_first' not in col) and ('_last' not in col)] + postllm_saturation_xrag_only
    postllm_attn_features = [col for col in df_features.columns.values if ('attn' in col) and (col[-2:] != '_n')]

    features_results = {}
    for feat_name, feat_list in [
        ('context', context_features),
        ('preproj', preproj_saturation_features),
        ('postproj', postproj_saturation_features),
        ('prellm', prellm_saturation_features),
        ('mid_xrag', mid_saturation_xrag_only),
        ('mid_joint', mid_saturation_with_nonxrag),
        ('mid_attn', mid_attn_features),
        ('last_xrag', last_saturation_xrag_only),
        ('last_joint', last_saturation_with_nonxrag),
        ('last_attn', last_attn_features),
        ('postllm_xrag', postllm_saturation_xrag_only),
        ('postllm_joint', postllm_saturation_with_nonxrag),
        ('postllm_attn', postllm_attn_features),
    ]:
        if not feat_list:
            continue
        required = feat_list + ['overflow_label']
        if not all(c in df_features.columns for c in required):
            continue
        data = df_features[required].dropna()
        if len(data) == 0:
            continue
        features_results[feat_name] = evaluate_feature_set(data[feat_list].values, data['overflow_label'].values)
    return features_results


In [19]:
# ============================================================================
# TABLES: MAIN RESULTS + INTERNAL ABLATION
# ============================================================================

def generate_tables():
    columns = COLUMN_ORDER

    all_data = {}
    for col_key in columns:
        cfg = get_column_cfg(col_key)
        all_data[col_key] = {
            'probing_no_query': load_probing_results(col_key, 'xrag_no_query'),
            'probing_setting1': load_probing_results(col_key, 'setting1'),
            'probing_setting2': load_probing_results(col_key, 'setting2'),
            'features': None,
        }

        feature_path = cfg.get("features_path")
        if feature_path:
            if os.path.exists(feature_path):
                df_features = load_dataset(feature_path)
                all_data[col_key]['features'] = compute_feature_results(df_features)
            else:
                print(f"[warn] Missing features file for {col_key}: {feature_path}")

    def fmt(mean, std):
        if mean is None or std is None:
            return "---"
        return f"{mean:.3f} ± {std:.3f}"

    def get_probe_metrics(col_key, key):
        for result_type in ['probing_no_query', 'probing_setting1', 'probing_setting2']:
            data = all_data[col_key][result_type]
            if data and key in data:
                result = data[key]
                return {
                    'auc_mean': result.get('auc'),
                    'auc_std': result.get('auc_std'),
                }
        return None

    def get_feature_metrics(col_key, key):
        if all_data[col_key]['features'] and key in all_data[col_key]['features']:
            return all_data[col_key]['features'][key]
        return None

    def build_table_df(sections_config):
        rows_display = []
        rows_means = []

        for stage_name, section_rows in sections_config:
            for row_idx, (feature_name, keys) in enumerate(section_rows):
                probe_key, feature_key = keys
                row_d = {'Stage': stage_name if row_idx == 0 else '', 'Features': feature_name}
                row_m = {'Stage': stage_name if row_idx == 0 else '', 'Features': feature_name}

                for col_key in columns:
                    col = COLUMN_LABELS[col_key]
                    m = get_probe_metrics(col_key, probe_key) if probe_key else (
                        get_feature_metrics(col_key, feature_key) if feature_key else None
                    )
                    if m and m.get('auc_mean') is not None:
                        row_d[col] = fmt(m['auc_mean'], m['auc_std'])
                        row_m[col] = m['auc_mean']
                    else:
                        row_d[col] = '---'
                        row_m[col] = np.nan

                rows_display.append(row_d)
                rows_means.append(row_m)

        return pd.DataFrame(rows_display), pd.DataFrame(rows_means)

    main_results_sections = [
        ("Pre-compression", [
            ("Context", [None, 'context']),
        ]),
        ("Pre-inference", [
            ("Embedding", ['xrag_preproj+postproj_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting1_preproj+postproj_with_preproj_q+postproj_q_linear_sklearn', None]),
            ("Saturation", [None, 'prellm']),
        ]),
        ("Post-inference", [
            ("Attention", [None, 'postllm_attn']),
            ("Embedding", ['xrag_mid+last_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting2_mid+last_with_mid_q+last_q_linear_sklearn', None]),
            ("Saturation", [None, 'postllm_xrag']),
            ("Saturation-joint", [None, 'postllm_joint']),
        ]),
    ]

    internal_ablation_sections = [
        ("Pre-projection", [
            ("Embedding", ['xrag_preproj_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting1_preproj_with_preproj_q_linear_sklearn', None]),
            ("Saturation", [None, 'preproj']),
        ]),
        ("Post-projection", [
            ("Embedding", ['xrag_postproj_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting1_postproj_with_postproj_q_linear_sklearn', None]),
            ("Saturation", [None, 'postproj']),
        ]),
        ("Middle layer", [
            ("Attention", [None, 'mid_attn']),
            ("Embedding", ['xrag_mid_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting2_mid_with_mid_q_linear_sklearn', None]),
            ("Saturation", [None, 'mid_xrag']),
            ("Saturation-joint", [None, 'mid_joint']),
        ]),
        ("Last layer", [
            ("Attention", [None, 'last_attn']),
            ("Embedding", ['xrag_last_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting2_last_with_last_q_linear_sklearn', None]),
            ("Saturation", [None, 'last_xrag']),
            ("Saturation-joint", [None, 'last_joint']),
        ]),
    ]

    table1_df, table1_means = build_table_df(main_results_sections)
    table2_df, table2_means = build_table_df(internal_ablation_sections)
    return (table1_df, table1_means), (table2_df, table2_means)


In [20]:
(table1_df, table1_means), (table2_df, table2_means) = generate_tables()

display(table1_df.style.apply(lambda _: style_auc_table(table1_df, table1_means, DATA_COLS), axis=None))
display(table2_df.style.apply(lambda _: style_auc_table(table2_df, table2_means, DATA_COLS), axis=None))


Loaded 7174 rows from /app/overflow-detection/scripts/data_preprocessing/runs/trivia_7b/features_context_metrics.jsonl
Loaded 4571 rows from /app/overflow-detection/scripts/data_preprocessing/runs/squad_7b/features_context_metrics.jsonl
Loaded 4571 rows from /app/overflow-detection/scripts/data_preprocessing/runs/hotpotqa_7b/features_context_metrics.jsonl
Loaded 16867 rows from /app/overflow-detection/scripts/data_preprocessing/runs/merged_all_no_llm_7b/features_context_metrics.jsonl


,Stage,Features,TriviaQA,SQuAD,HotpotQA,SQuAD + Trivia + Hotpot
0,Pre-compression,Context,0.607 ± 0.008,0.592 ± 0.020,0.592 ± 0.020,0.497 ± 0.013
1,Pre-inference,Embedding,0.684 ± 0.020,0.634 ± 0.010,0.640 ± 0.008,0.727 ± 0.009
2,,Embedding-joint,0.717 ± 0.022,0.673 ± 0.019,0.694 ± 0.009,0.758 ± 0.012
3,,Saturation,0.538 ± 0.011,0.529 ± 0.029,0.529 ± 0.029,0.592 ± 0.011
4,Post-inference,Attention,0.628 ± 0.011,0.606 ± 0.007,0.606 ± 0.007,0.660 ± 0.008
5,,Embedding,0.680 ± 0.023,0.635 ± 0.011,0.639 ± 0.007,0.724 ± 0.009
6,,Embedding-joint,0.712 ± 0.026,0.688 ± 0.015,0.707 ± 0.009,0.768 ± 0.010
7,,Saturation,0.576 ± 0.011,0.526 ± 0.021,0.526 ± 0.021,0.613 ± 0.007
8,,Saturation-joint,0.573 ± 0.019,0.590 ± 0.017,0.590 ± 0.017,0.669 ± 0.011


,Stage,Features,TriviaQA,SQuAD,HotpotQA,SQuAD + Trivia + Hotpot
0,Pre-projection,Embedding,0.677 ± 0.017,0.621 ± 0.011,0.618 ± 0.014,0.718 ± 0.009
1,,Embedding-joint,0.699 ± 0.022,0.656 ± 0.013,0.677 ± 0.012,0.747 ± 0.013
2,,Saturation,0.516 ± 0.014,0.512 ± 0.017,0.512 ± 0.017,0.499 ± 0.010
3,Post-projection,Embedding,0.671 ± 0.025,0.629 ± 0.011,0.637 ± 0.008,0.720 ± 0.008
4,,Embedding-joint,0.709 ± 0.025,0.670 ± 0.016,0.687 ± 0.010,0.752 ± 0.012
5,,Saturation,0.539 ± 0.017,0.531 ± 0.025,0.531 ± 0.025,0.591 ± 0.011
6,Middle layer,Attention,0.611 ± 0.010,0.586 ± 0.014,0.586 ± 0.014,0.645 ± 0.008
7,,Embedding,0.671 ± 0.025,0.629 ± 0.011,0.637 ± 0.008,0.720 ± 0.008
8,,Embedding-joint,0.704 ± 0.027,0.683 ± 0.014,0.702 ± 0.010,0.763 ± 0.011
9,,Saturation,0.539 ± 0.017,0.531 ± 0.025,0.531 ± 0.025,0.591 ± 0.011
